# Digital Forensics - investigação do endpoint `/invoices/search`

Notebook reproduzível para preservação, parsing, análise e exportação de resultados. A evidência bruta e as saídas restritas não são versionadas. O formato de timestamp da fonte é `YYYY-DD-MMTHH:MM`, apesar da aparência ISO; o pipeline faz o parsing explícito.

## Hipóteses

- H1: houve automação em massa para consultar objetos de outros usuários.
- H2: tokens foram reutilizados entre origens e invoices em padrão incompatível com uso humano.
- H3: respostas de sucesso indicam um conjunto potencialmente exposto.
- H4: o volume aumentou ao longo do trimestre.

Limite: sem owner/account ID, response body e trilha de autorização não se prova acesso cross-account nem entrega de PII.

In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from meli_forensics.evidence import resolve_evidence, sha256_file
from meli_forensics.pipeline import build_database, export_results, build_charts

INPUT = Path(os.environ.get('FORENSIC_INPUT', ROOT / 'evidence' / 'three_months_logs.zip'))
WORK = ROOT / 'work'
PUBLIC = ROOT / 'results' / 'public'
RESTRICTED = ROOT / 'results' / 'restricted'
INPUT

## Integridade e ingestão

Defina `FORENSIC_INPUT` para o ZIP ou CSV. O hash SHA-256 é calculado antes da análise.

In [ ]:
evidence = resolve_evidence(INPUT, WORK / 'evidence')
integrity = {'path': str(evidence), 'sha256': sha256_file(evidence)}
database = WORK / 'forensics.duckdb'
integrity.update(build_database(evidence, database))
integrity

## Análise e enriquecimento

As saídas públicas mascaram IPs e substituem tokens por fingerprints SHA-256. A pasta restrita contém os rankings completos e está no `.gitignore`.

In [ ]:
summary = export_results(database, PUBLIC, RESTRICTED, WORK / '.ipwho_cache.json')
build_charts(PUBLIC, PUBLIC / 'charts')
summary

## Resultado executado desta entrega (sanitizado)

- Linhas: **4,478,619**
- Janela: **2025-10-01T00:00:00 a 2025-12-31T23:59:00**
- IPs automatizados de alta confiança: **5**
- Requisições do cluster: **61,961**
- Invoices distintos: **10,181**
- HTTP 200: **44,868** em **10,173** invoices.

In [ ]:
import csv
def read_result(name):
    with (PUBLIC / name).open(encoding='utf-8') as f:
        return list(csv.DictReader(f))

top_20_ips = read_result('top_20_ips.csv')
top_10_tokens = read_result('top_10_tokens.csv')
site_impact = read_result('site_impact.csv')
top_20_ips[:5], top_10_tokens[:3], site_impact

## Conclusão

O conjunto mostra exploração automatizada compatível com IDOR em alta confiança. A conclusão sobre divulgação de PII permanece em confiança média: HTTP 200 delimita um conjunto conservador potencialmente exposto, mas é necessário correlacionar response body, identidade do token, autorização e auditoria de dados.